# 4. Fair model comparison and reporter-grounded distillation

## Goal

Can a deployable learner inherit useful metabolic structure from a computational teacher without hidden-state leakage, and does that improve on matched baselines?

This notebook is a readable research record. It follows the actual hand-offs in order and loads the saved evidence by default; it does **not** hide the experiment behind a one-cell runner.


## Pipeline at a glance

```text
goal → declared generator → observable/lockbox split → model setup & training
     → candidate or condition screen → matched comparison → interpretation
```

Each section below corresponds to one of these hand-offs.


In [ ]:
# Run this notebook from the repository root.
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

REGENERATE = False  # Cached artifacts are the default; no expensive solve runs implicitly.

def artifact(relative_path: str) -> Path:
    """Fail with a useful message rather than silently replacing evidence."""
    path = ROOT / relative_path
    if not path.exists():
        raise FileNotFoundError(f"Missing cached artifact: {path}")
    return path

def show(frame, n=8):
    # `print` keeps this notebook usable in a plain Python kernel as well as Jupyter.
    print(frame.head(n).to_string(index=False))
    print(f"{len(frame):,} rows × {len(frame.columns):,} columns")


## 1. Experimental contract

The experiment has a declared observation boundary. “Observable” means the learner may use it; “lockbox” means it may be generated and audited but must not be used as a deployable feature.


In [ ]:
from yeast_validation import run_reporter_grounded_hybrid_distillation as distill
from yeast_validation import gem_gsm_surrogate as surrogate

print("student-visible controls:", distill.GEM_APPLIED_CONTROL_COLUMNS)
print("student-visible reporters:", distill.REPORTER_COLUMNS)
print("surrogate output channels:", surrogate.SURROGATE_OUTPUT_COLUMNS)


## 2. Data generator

The dynamic-GEM cultures from the preceding phase provide observable trajectories and a separately declared teacher target set. The notebook keeps train/test culture identity and channel availability visible.

The next cell exposes the generator’s first concrete hand-off. It is deliberately small/inspection-only where generating the full campaign is expensive.


In [ ]:
# Load the declared training interface rather than calling an opaque experiment wrapper.
targets = pd.read_csv(artifact("data/gem_dynamic_capacity_constraints.csv"))
visible = [c for c in distill.REPORTER_COLUMNS + ["temperature", "pH", "DO"] if c in targets]
show(targets[[c for c in ["culture_id", "interval_index"] + visible if c in targets]])


## 3. Model setup and training contract

Teacher and student are separate: the teacher/surrogate may use declared metabolic targets in training; the deployed student gets only the observation interface. The student’s control trajectory is then evaluated through the GSM surrogate or exact replay.

Training is not automatically started in this notebook. The cached training/evaluation artifacts below are the evidence record; regeneration must be an intentional, parameterized action.


In [ ]:
# Make the experiment hand-off inspectable before looking at aggregate metrics.
for name, relative_path in [('metrics', 'data/hybrid_student_metrics.csv')]:
    path = artifact(relative_path)
    print(f"{name}: {path.relative_to(ROOT)}")


## 4. Screening / selection stage

Candidate/model screening uses the same split and observation contract. Cached exact replay is kept separate from cheap surrogate ranking.

The screen is intentionally shown separately from final verification, so a virtual score cannot be mistaken for an exact outcome.


In [ ]:
# Load the primary evidence table and inspect its schema before aggregation.
metrics = pd.read_csv(artifact('data/hybrid_student_metrics.csv'))
show(metrics)


## 5. Matched comparison

Compare student variants, baselines, clean-teacher ablations, and observation-noise controls—not a pooled metric from unequal inputs.


In [ ]:
# Aggregate only over fields that exist in this version of the cached record.
comparison = metrics.groupby(['model', 'split'], dropna=False).mean(numeric_only=True)
show(comparison.reset_index() if hasattr(comparison, "reset_index") else comparison)


## 6. Analysis view

The plot is intentionally generic: it exposes every numeric evidence column so the reader can select the metric relevant to the claim, rather than hard-coding an attractive subset.


In [ ]:
numeric = metrics.select_dtypes("number")
if numeric.shape[1]:
    ax = numeric.plot(kind="box", rot=45, figsize=(11, 4), title="Cached evidence: numeric metric distribution")
    ax.set_ylabel("recorded metric value")
    plt.tight_layout()
else:
    print("This artifact has no numeric columns to plot.")


## 7. Interpretation, scope, and next hand-off

A good surrogate ranking is not an exact-GEM guarantee. The notebook therefore presents trajectory metrics and replay/verification evidence as distinct claims.

### Reproduction boundary

The cells above reveal the inputs and artifacts without launching an expensive campaign. To regenerate, use the explicit command below only after reviewing its declared inputs and output destination.


In [ ]:
if REGENERATE:
    # This guard prevents accidental solver/campaign execution.
    raise RuntimeError('Run the explicitly parameterized distillation pipeline from its documented configuration; this notebook does not hide it behind run_all().')
